# timeseries, correcting for rectification

## Imports

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## compute indices

In [ ]:
def get_nino3(x, z_t=70, lat_bound=5):
    """get Niño 3 average, with checks in place"""

    if "z_t" in x.dims:
        x = x.sel(z_t=z_t, method="nearest")

    if "latitude" in x.dims:
        x = x.sel(latitude=slice(-lat_bound, lat_bound)).mean("latitude")

    return x.sel(longitude=slice(210, 270)).mean("longitude")


def get_nino3_helper(x, lat_bounds):
    """get Niño 3 average, with checks in place"""

    ## get index to select data
    idx = dict(latitude=slice(*lat_bounds), longitude=slice(210, 270))

    return x.sel(idx).mean(["longitude", "latitude"])


def get_dTdz_sub(Tsub, mld=50, delta=25):
    """get velocity at base of mixed layer"""

    ## get temperature difference
    dT = src.utils.get_dT_sub(Tsub=Tsub, Hm=mld, delta=delta)

    ## get gradient
    dTdz = -dT / mld

    return dTdz

### individual metrics

#### $h$ max-gradient stuff

In [ ]:
# ## load tropical SST avg
trop_sst = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th_total = xr.merge([Th_total, trop_sst])
Th_anom = Th_total - Th_total.mean("member")

## custom h data
h_mg_forced, h_mg_anom = src.utils.load_h_data(max_grad=True)
h_total = h_mg_forced + h_mg_anom

## compute h stats
h = h_total.sel(longitude=slice(120, 280)).mean("longitude")
h_e = h_total.sel(longitude=slice(210, 270)).mean("longitude")
h_w = h_total.sel(longitude=slice(120, 180)).mean("longitude")
h_idxs = xr.merge([h.rename("h_mg"), (h_w - h_e).rename("dhdx_mg")])

#### Niño 3 averages

In [ ]:
## load spatial data
forced, anom_ = src.utils.load_consolidated()

## get subset of data for total
VARNAMES = ["T", "w", "sst", "taux"]
total = anom_[VARNAMES] + forced[VARNAMES]
total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])

## get nino3 averages
total_n3 = src.utils.reconstruct_wrapper(total, fn=get_nino3)

## rename to avoid conflicts
total_n3 = total_n3.rename({v: f"{v}_n3" for v in VARNAMES})

## compute stratification
total_n3["dTdz_n3"] = src.utils.reconstruct_wrapper(
    total[["T", "T_comp"]],
    lambda x: get_dTdz_sub(x.sel(longitude=slice(210, 270)).mean("longitude")),
)["T"]

#### $\partial T / \partial x$

In [ ]:
## compute diff. def of Tw
get_Tw = lambda x: x.sel(latitude=slice(-5, 5), longitude=slice(120, 160)).mean(
    ["longitude", "latitude"]
)
Th_total["Tw"] = src.utils.reconstruct_wrapper(total[["sst", "sst_comp"]], get_Tw)[
    "sst"
]

## compute dTdx
Th_total["dTdx"] = Th_total["Tw"] - Th_total["T_3"]

#### $\tau_x$

In [ ]:
def get_taux(x):
    """function to compute area-averaged taux"""

    ## specify lon/lat range for averaging
    lon_range = slice(130, 220)
    lat_range = slice(-5, 5)
    idx = dict(longitude=lon_range, latitude=lat_range)

    ## compute
    return x.sel(idx).mean(["longitude", "latitude"])


## compute
taux = src.utils.reconstruct_wrapper(total[["taux", "taux_comp"]], fn=get_taux)

### Combine data

In [ ]:
## combine data
data = xr.merge([Th_total, total_n3, h_idxs, taux, Th_anom["T_34"].rename("T_34a")])

## get windowed version
data = src.utils.get_windowed(
    data.isel(time=slice(None, -1)), window_size=480, stride=60
)

## resample to DJF
data = data.resample({"time": "QS-DEC"}).mean().isel(time=slice(4, None, 4))

## compute ensemble mean and sigma
data_mean = data.mean("time")
data_sigma = data.std("time")

#### Look at subset

In [ ]:
## specify year index
yi = 7

fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")
for ax, v in zip(axs, ["w_n3", "dTdx", "dTdz_n3", "taux"]):
    ax.set_title(v)
    ax.scatter(
        data_sigma["T_34a"].isel(year=yi),
        data_mean[v].isel(year=yi),
    )

### Remove nonlinear rect. effect

In [ ]:
## Linear regression object
LR = sklearn.linear_model.LinearRegression

## empty array to hold results
coefs = []

## loop thru years
for year in data.year:

    ## get data
    X = data_sigma["T_34a"].sel(year=year)
    # Y = data_mean["dTdx"].sel(year=year)
    Y = data_mean.sel(year=year).to_dataarray().transpose("member", ...)

    ## instantiate model
    mod = LR(fit_intercept=True)
    mod.fit(X=X.values[:, None], y=Y.values)

    ## get coefficients
    coefs_y = np.concatenate([mod.coef_, mod.intercept_[:, None]], axis=1)

    ## put in xr
    coefs_y = xr.DataArray(
        coefs_y,
        coords=dict(variable=Y["variable"], degree=[1, 0]),
        dims=["variable", "degree"],
    )

    coefs.append(coefs_y.to_dataset("variable"))

## put coefs in array
coefs = xr.concat(coefs, dim=data.year)

In [ ]:
for v in ["w_n3", "dTdx", "dTdz_n3", "h_mg", "dhdx_mg"]:
    fig, ax = plt.subplots(figsize=(3, 2.5))
    ax.plot(data.year, coefs.sel(degree=0)[v])
    ax.plot(data.year, data[v].mean(["member", "time"]))
    ax.axvline(2010, c="gray", lw=1, ls="--")
    ax.set_title(v)
    plt.show()

## Make plots

## Old

Reference plot

In [ ]:
sel_ = lambda x: x.mean("month")
sel = lambda x: (sel_(x) - sel_(x.isel(year=0))) / sel_(x.isel(year=0))

fig, ax = plt.subplots(figsize=(3.5, 3), layout="constrained")
ax.plot(forced.year, sel(dTdz_n3), label=r"$\Delta_z T$")
# ax.plot(forced.year, sel(forced["T_4"] - forced["T_3"]), label=r"$\Delta_x T$")
ax.plot(forced.year, sel(forced["Tw"] - forced["T_3"]), label=r"$\Delta_x T$")
ax.plot(forced.year, sel(H_mg_stats["Hbar"]), label=r"$h$")
ax.plot(forced.year, sel(forced["w_n3"]), label=r"$w$")


## format/labelling
ax.set_yticks([-0.3, 0, 0.6])
ax.axhline(0, **ref_kwargs)
format_ax(ax)
ax.legend()
ax.set_ylabel("Fractional change")
ax.set_title(r"(e) Fractional changes", loc="left")

## save
# save(fig, "mean-state_timeseries")

plt.show()